In [9]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [11]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [13]:
# preparing the dataset
import pandas as pd
dataset_path = '/content/paired_dataset_clean_10k.csv'
humanized_df = pd.read_csv(dataset_path)
humanized_df.head(1)

In [14]:
train_prompt_style = """
### Instruction:
Convert the following AI-generated text into natural human-like text.

### Input:
{}

### Response:
{}
"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN to signal the end of each example

def formatting_prompts_func(examples):
    inputs = examples["ai_text"]        # ✅ AI text is input
    outputs = examples["human_text"]    # ✅ Human text is output
    texts = []

    for input_text, output_text in zip(inputs, outputs):
        text = train_prompt_style.format(input_text, output_text) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

# Apply to dataset
from datasets import Dataset
train_dataset = Dataset.from_pandas(humanized_df[['ai_text', 'human_text']])

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)


print("Dataset formatted ✅")
print("\nSample:")
print(train_dataset[0]['text'][:300])

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for 1 full training run.
        max_steps = 500,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

In [16]:
trainer_stats = trainer.train()

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [19]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
ai_text = '''
Love is a complex and multifaceted phenomenon that emerges from the interplay of emotional, cognitive, and behavioral processes, characterized by a deep sense of attachment, empathy, and mutual understanding between individuals. It operates as both a psychological state and a social construct, often reinforced through consistent patterns of care, trust, and shared experiences. From an analytical perspective, love can be interpreted as a dynamic system that evolves over time, influenced by internal variables such as personal values and external factors like environment and interaction frequency. Despite its abstract nature, it manifests in observable actions—support, sacrifice, and communication—making it both measurable in behavior and immeasurable in depth.

'''
inputs = tokenizer(
[
    train_prompt_style.format(

        ai_text,
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to(device)

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

In [22]:
# Switch to inference mode — required by Unsloth
model.to("cuda") # Explicitly move model to GPU
FastLanguageModel.for_inference(model)

# Inference prompt template — output left blank for generation
inference_prompt = """### Instruction:
Convert the following AI-generated text into natural human-like text.

### Input:
{}

### Response:
"""

ai_text = """Love is a complex and multifaceted phenomenon that emerges from the interplay \
of emotional, cognitive, and behavioral processes, characterized by a deep sense of attachment, \
empathy, and mutual understanding between individuals. It operates as both a psychological state \
and a social construct, often reinforced through consistent patterns of care, trust, and shared \
experiences. From an analytical perspective, love can be interpreted as a dynamic system that \
evolves over time, influenced by internal variables such as personal values and external factors \
like environment and interaction frequency. Despite its abstract nature, it manifests in observable \
actions—support, sacrifice, and communication—making it both measurable in behavior and immeasurable in depth."""

inputs = tokenizer(
    [inference_prompt.format(ai_text.strip())],
    return_tensors = "pt"
).to("cuda")   # always explicitly send to "cuda", not a device variable

from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt=True)  # skip_prompt hides the input, shows only output

print("\n--- MODEL OUTPUT ---\n")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.7,     # slight randomness for more natural text
    do_sample = True,
    repetition_penalty = 1.1  # discourages repetitive phrasing
)


In [26]:
!pip install sentencepiece peft bitsandbytes -q

In [21]:
model.save_pretrained("HumanizerModel")
tokenizer.save_pretrained("HumanizerTokenizer")
print("Model saved ✅")

In [27]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# ============================================================
# LOAD
# ============================================================
model_path     = "/content/HumanizerModel"
tokenizer_path = "/content/HumanizerTokenizer"

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/gemma-2-2b-it-bnb-4bit",
    quantization_config = bnb_config,
    dtype = torch.float16,
    device_map = "auto",
)

model = PeftModel.from_pretrained(base_model, model_path)
model.eval()
print("Model loaded ✅")


# ============================================================
# INFERENCE
# ============================================================
inference_prompt = """### Instruction:
Convert the following AI-generated text into natural human-like text.

### Input:
{}

### Response:
"""

ai_text = """Love is a complex and multifaceted phenomenon that emerges from the interplay
of emotional, cognitive, and behavioral processes, characterized by a deep sense of attachment,
empathy, and mutual understanding between individuals."""

inputs = tokenizer(
    [inference_prompt.format(ai_text.strip())],
    return_tensors = "pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,
        do_sample = True,
        temperature = 0.7,
        repetition_penalty = 1.1,
        use_cache = True,
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))

In [30]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/HumanizerModel",
    tokenizer_name = "/content/HumanizerTokenizer",
    max_seq_length = 768,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

inference_prompt = """### Instruction:
Convert the following AI-generated text into natural human-like text without adding and removing existing meaning of sentence.

### Input:
{}

### Response:
"""

ai_text = """

Artificial General Intelligence (AGI) refers to a highly advanced form of artificial intelligence that possesses the ability to understand, learn, and apply knowledge across a wide range of tasks at a level comparable to or exceeding that of human intelligence. Unlike narrow AI, which is designed for specific tasks such as image recognition or language translation, AGI is characterized by its adaptability, reasoning capabilities, and capacity for autonomous problem-solving in unfamiliar domains. It integrates various cognitive functions, including perception, memory, decision-making, and creativity, enabling it to generalize knowledge and transfer learning between different contexts. While AGI remains largely theoretical at present, ongoing research in machine learning, neural networks, and cognitive architectures continues to push the boundaries toward its potential realization, raising important considerations regarding ethics, safety, and societal impact.


"""

inputs = tokenizer(
    [inference_prompt.format(ai_text.strip())],
    return_tensors = "pt"
).to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs,
    streamer = streamer,
    max_new_tokens = 256,
    temperature = 0.7,
    do_sample = True,
    repetition_penalty = 1.1,
)